# EDA: norfolk_doc_observed_events

This notebook is **toolkit-seeded** and explore-ready. Every `eda()` run writes a new notebook (`eda.ipynb`, then `eda_1.ipynb`, ...) and never modifies an existing one, so anything you author here is safe; open the newest file to pick up refreshed seed cells. Run the loader cell first to bind the experiment variables, then edit/add cells freely; your EDA never modifies git-tracked `hhemt` source.

In [ ]:
from pathlib import Path

from hhemt.eda import load_eda_context, render_eda_plots

root = Path.cwd()
ctx = load_eda_context(root)
ctx.is_bundle, ctx.cfg_analysis.analysis_id

## Variable inventory

`ctx` exposes: `datatree`, `sensitivity_datatree`, `cfg_analysis`, `cfg_system`, `scenario_status`, `swmm_features`, `triton_dem`, `performance`, `is_bundle`. Fields are `None` when the experiment or this root does not carry that artifact.

In [ ]:
{k: (type(v).__name__ if v is not None else None) for k, v in vars(ctx).items()}

In [ ]:
# Seed figures: re-rendered on execution from the carried EDA datasets and
# displayed inline (the notebook is the main explore surface).
# Guarded on the EDA data-prep zarr stores being present at root (absent on a
# figureless non-sensitivity analysis or a bundle whose eda/ zarr was not
# harvested) so this cell never errors.
from hhemt.eda import _report as _eda_report
from hhemt.eda._plotting import _RENDERER_BACKING_ARTIFACT

# One builder per enabled plot kind, gated per kind on that kind's own backing
# zarr. A root-level any('*.zarr') guard passes whenever any eda artifact exists
# (byte identity always writes one), so it would still run a builder against an absent
# store whenever its own calc member legitimately skipped.
_BUILDERS = {
    "config_diff_maps": _eda_report.config_diff_maps_figure_from_root,
    "dem_resolution_cost_error": _eda_report.dem_resolution_cost_error_figure_from_root,
    "dem_resolution_error_ecdf": lambda r: _eda_report.dem_resolution_error_ecdf_figure_from_root(
        r, eda_cfg=ctx.cfg_analysis.eda
    ),
    "dem_resolution_diff_maps": _eda_report.dem_resolution_diff_maps_figure_from_root,
    "dem_resolution_coupling_table": _eda_report.dem_resolution_coupling_table_figure_from_root,
}
_eda_dir = root / "eda"


def _backed(k):
    return (_eda_dir / f"{_RENDERER_BACKING_ARTIFACT.get(k, k)}.zarr").is_dir()


_kinds = [k for k in ctx.cfg_analysis.eda.enabled_plots if k in _BUILDERS and _backed(k)]
_figs = [_BUILDERS[k](root) for k in _kinds]
# also persist the standalone plots/eda/*.html artifacts (re-render to disk)
plot_paths = render_eda_plots(root, cfg_analysis=ctx.cfg_analysis, eda_cfg=ctx.cfg_analysis.eda) if _figs else []
for _f in _figs:
    _f.show()
# The final expression is a summary string, not a figure: a bare trailing
# `_figs[0]` re-displays the figure a second time (nbconvert emits output
# from both `.show()` and the cell's last expression), doubling every
# figure in eda_report.html.
(
    f"{len(_figs)} EDA {'figure' if len(_figs) == 1 else 'figures'} rendered inline above."
    if _figs
    else "no EDA figures for this experiment"
)

## Byte identity across compute configurations (source-analysis only)

Available only on a live analysis directory (a transported bundle carries no source per-scenario summaries).